In [0]:
# 1. IMPORTS AND SETTINGS

import os
from pyspark.sql import functions as F

# Target Unity Catalog Volume where raw files landed
volume_base_path = "/Volumes/7trents_data_platform/bronze/raw_data"

# Dimensions are available in the Day 1 folder
batch_date_str = "01_04_2026"
source_path = os.path.join(volume_base_path, batch_date_str)

# Target Bronze schema in 7trents_data_platform
target_schema = "7trents_data_platform.bronze"

print("Dimension source path:", source_path)
print("Target schema:", target_schema)


In [0]:


# 2. CREATE DIMENSION INGESTION FUNCTION

def dimension_ingestor(source_name):

    # 1. Define CSV file path from Volume
    file_path = os.path.join(source_path, f"{source_name}.csv")

    print(f"\nReading: {file_path}")

    # 2. Read CSV file directly from the volume
    df = (spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load(file_path)
    )

    # 3. Add audit columns
    df = (df
        .withColumn("source_file", F.lit(f"{source_name}.csv"))
        .withColumn("load_date", F.to_date(F.lit(batch_date_str), "dd_MM_yyyy"))
        .withColumn("ingestion_ts", F.current_timestamp())
        .withColumn("load_type", F.lit("FULL"))
    )

    print(f"Total records loaded: {df.count()}")
    return df

# 3. TEST DIMENSION INGESTION

df_people = dimension_ingestor("DimPeople")
df_people.show(5, truncate=False)


In [0]:



# 4. LOAD ALL DIMENSION TABLES TO BRONZE

# Source CSV names mapped to target Bronze Delta table names
dimension_tables = {
    "DimPeople": "dim_people",
    "DimCounty": "dim_county",
    "DimCancellationReason": "dim_cancellation_reason",
    "DimShrinkageType": "dim_shrinkage_type"
}

# Process each dimension table
for source_name, table_name in dimension_tables.items():

    target_table = f"{target_schema}.{table_name}"

    # Check if table already exists
    if spark.catalog.tableExists(target_table):
        print(f"Table '{target_table}' already exists; skipping.")
    else:
        # Read data from Volume using the ingestion function
        df = dimension_ingestor(source_name)

        # Save as Delta table in the bronze schema
        (df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(target_table)
        )

        print(f"Table '{target_table}' created and saved successfully.")



In [0]:
# we have already created this mapping we are now just checking
# Source CSV names mapped to target Bronze Delta table names
dimension_tables = {
    "DimPeople": "dim_people",
    "DimCounty": "dim_county",
    "DimCancellationReason": "dim_cancellation_reason",
    "DimShrinkageType": "dim_shrinkage_type"
}

# 5. VALIDATE ALL DIMENSION TABLES

for table_name in dimension_tables.values():

    target_table = f"{target_schema}.{table_name}"

    df = spark.table(target_table)

    print("\n-------------------------------------------")
    print("Table:", target_table)
    print("Total records:", df.count())
    df.groupBy("load_date", "load_type").count().show()